In [11]:
import streamlit as st
from pypdf import PdfReader
import re
import numpy as np
import chromadb
import ollama

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


st.set_page_config(
    page_title="RAG Document Q&A",
    page_icon="📚",
    layout="wide"
)

st.title("📚 RAG-Based Document Question Answering")
st.write("Upload a PDF and ask questions about its content.")

# =====================================================
# LOAD MODELS
# =====================================================

@st.cache_resource
def load_models():

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

    reranker = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )

    return embedding_model, reranker


embedding_model, reranker = load_models()


# =====================================================
# RRF FUNCTION
# =====================================================

def reciprocal_rank_fusion(rankings, k=60):

    scores = {}

    for ranking in rankings:

        for rank, doc_id in enumerate(ranking):

            if doc_id not in scores:
                scores[doc_id] = 0

            scores[doc_id] += 1 / (k + rank + 1)

    sorted_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_results


# =====================================================
# UPLOAD PDF
# =====================================================

uploaded_file = st.file_uploader(
    "Upload your PDF",
    type=["pdf"]
)


if uploaded_file is not None:

    # =================================================
    # EXTRACT TEXT
    # =================================================

    reader = PdfReader(uploaded_file)

    all_text = ""

    for page in reader.pages:

        text = page.extract_text()

        if text:
            all_text += text + "\n"

    st.success("Document uploaded successfully!")

    st.write("Number of pages:", len(reader.pages))


    # =================================================
    # CLEAN TEXT
    # =================================================

    clean_text = re.sub(
        r'\s+',
        ' ',
        all_text
    ).strip()


    # =================================================
    # CHUNKING
    # =================================================

    words = clean_text.split()

    chunk_size = 500
    overlap = 50

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += chunk_size - overlap


    st.write("Number of chunks:", len(chunks))


    # =================================================
    # CREATE EMBEDDINGS
    # =================================================

    with st.spinner("Creating embeddings..."):

        chunk_embeddings = embedding_model.encode(
            chunks,
            show_progress_bar=False
        )

    st.success("Embeddings created successfully!")


    # =================================================
    # CHROMADB
    # =================================================

    client = chromadb.Client()

    collection = client.get_or_create_collection(
        name="rag_documents"
    )

    ids = [
        f"chunk_{i}"
        for i in range(len(chunks))
    ]

    metadatas = [
        {
            "chunk_id": i,
            "source": uploaded_file.name
        }
        for i in range(len(chunks))
    ]

    # Remove old chunks if they exist
    try:

        collection.delete(
            ids=ids
        )

    except:

        pass


    collection.add(
        ids=ids,
        documents=chunks,
        embeddings=chunk_embeddings.tolist(),
        metadatas=metadatas
    )


    # =================================================
    # BM25
    # =================================================

    tokenized_chunks = [
        chunk.lower().split()
        for chunk in chunks
    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )


    st.success("Document is ready for questions!")


    # =================================================
    # QUESTION INPUT
    # =================================================

    st.subheader("Ask a Question")

    question = st.text_input(
        "Enter your question:"
    )


    # =================================================
    # ASK QUESTION
    # =================================================

    if st.button("🔍 Ask Question"):

        if question.strip() == "":

            st.warning(
                "Please enter a question."
            )

        else:

            # =========================================
            # VECTOR SEARCH
            # =========================================

            with st.spinner("Searching the document..."):

                question_embedding = (
                    embedding_model
                    .encode([question])
                    .tolist()
                )

                chroma_results = collection.query(
                    query_embeddings=question_embedding,
                    n_results=min(10, len(chunks))
                )

                vector_ids = chroma_results["ids"][0]


                # =====================================
                # BM25 SEARCH
                # =====================================

                query_tokens = question.lower().split()

                bm25_scores = bm25.get_scores(
                    query_tokens
                )

                bm25_indices = np.argsort(
                    bm25_scores
                )[::-1][:10]

                bm25_ids = [
                    f"chunk_{i}"
                    for i in bm25_indices
                ]


                # =====================================
                # RRF
                # =====================================

                rrf_results = reciprocal_rank_fusion(
                    [
                        vector_ids,
                        bm25_ids
                    ]
                )

                top_rrf_ids = [
                    doc_id
                    for doc_id, score in rrf_results[:5]
                ]


                # =====================================
                # GET CHUNKS
                # =====================================

                candidate_items = []

                for doc_id in top_rrf_ids:

                    index = int(
                        doc_id.split("_")[1]
                    )

                    candidate_items.append(
                        (
                            doc_id,
                            chunks[index]
                        )
                    )


                # =====================================
                # RERANKING
                # =====================================

                pairs = [
                    [question, chunk]
                    for doc_id, chunk
                    in candidate_items
                ]

                rerank_scores = reranker.predict(
                    pairs
                )

                reranked_items = sorted(
                    zip(
                        candidate_items,
                        rerank_scores
                    ),
                    key=lambda x: x[1],
                    reverse=True
                )

                top_items = reranked_items[:3]


            # =========================================
            # CREATE CONTEXT
            # =========================================

            citation_context = ""

            for i, ((doc_id, chunk), score) in enumerate(
                top_items
            ):

                citation_context += f"""

[Source {i + 1}]

Document: {uploaded_file.name}

Chunk ID: {doc_id}

Content:
{chunk}
"""


            # =========================================
            # OLLAMA PROMPT
            # =========================================

            prompt = f"""
You are a document question answering assistant.

Answer the question using ONLY the information
provided in the sources below.

Rules:

1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not present in the sources, say:

"I could not find this information in the document."

4. Include [Source 1], [Source 2], etc.
after important statements.
5. Keep the answer clear and concise.

Sources:

{citation_context}

Question:

{question}

Answer:
"""


            # =========================================
            # OLLAMA
            # =========================================

            with st.spinner("Generating answer..."):

                response = ollama.chat(
                    model="llama3.2",
                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]
                )

                answer = response["message"]["content"]


            # =========================================
            # DISPLAY ANSWER
            # =========================================

            st.subheader("💡 Answer")

            st.write(answer)


            # =========================================
            # DISPLAY SOURCES
            # =========================================

            st.subheader("📚 Sources")

            for i, ((doc_id, chunk), score) in enumerate(
                top_items
            ):

                with st.expander(
                    f"Source {i + 1} — {doc_id}"
                ):

                    st.write(
                        f"Document: {uploaded_file.name}"
                    )

                    st.write(
                        f"Reranker score: {score:.4f}"
                    )

                    st.write(chunk)

IndentationError: unexpected indent (3177916485.py, line 175)

^C


In [4]:
import os
print(os.getcwd())

C:\Users\praja\Desktop\RAG


In [14]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\praja\Desktop\RAG
['.gitignore', '.ipynb_checkpoints', 'chroma_db', 'cn project report (1).pdf', 'rag.ipynb', 'rag_app.py', 'README.md', 'requirements.txt', 'Untitled.ipynb']


In [13]:
os.getcwd()

'C:\\Users\\praja\\Desktop\\RAG'

In [15]:
import os
print(os.path.abspath("rag.ipynb"))

C:\Users\praja\Desktop\RAG\rag.ipynb
